# Vanishing & exploding gradients

_Deep Learning_

**Over long sequences, gradients shrink to nothing or blow up; clipping caps the blow-ups.**

Backprop through a long sequence multiplies many slopes together.

     If those slopes are small, the product shrinks toward 0: a vanishing gradient. The network can't learn long-range links.

---

This notebook is a practice scaffold for the **Vanishing & exploding gradients** lesson. We'll watch gradients vanish layer by layer, then confirm the effect on a real digit classifier. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Build a deep stack of sigmoid layers

We stack 20 `Linear` + `Sigmoid` layers. The sigmoid's slope is at most 0.25, so every layer backprop passes through multiplies the gradient by a small factor. With 20 layers those small factors compound, which is exactly the setup that makes gradients vanish.

In [ ]:
import torch
import torch.nn as nn

# 20 sigmoid layers: gradients shrink as they flow back
layers = []
for _ in range(20):
    layers += [nn.Linear(10, 10), nn.Sigmoid()]
net = nn.Sequential(*layers)

### Step 2 — Run a forward + backward pass

We push a small random batch through the network, collapse the output to a single number with `.sum()` so we have a scalar to differentiate, and call `.backward()`. After this, every layer's weight tensor carries a `.grad` holding its gradient.

In [ ]:
x = torch.randn(4, 10)
loss = net(x).sum()
loss.backward()

### Step 3 — Compare gradients at the two ends

`net[0]` is the first layer (nearest the input); `net[-2]` is the last `Linear` (nearest the output, just before the final sigmoid). The gradient near the output is healthy, but by the time it has been multiplied back through every layer to the first one it has shrunk dramatically — it has *vanished*.

In [ ]:
first = net[0].weight.grad.abs().mean().item()    # gradient near the input
last = net[-2].weight.grad.abs().mean().item()     # gradient near the output
print("grad at last  layer:", round(last, 8))
print("grad at first layer:", round(first, 8))     # far smaller -> vanished

### Step 4 — Clip the gradients to cap explosions

Vanishing is one failure mode; the opposite is *exploding* gradients, where the product blows up. `clip_grad_norm_` rescales all gradients so their combined norm never exceeds `max_norm`, a cheap safeguard that keeps a single huge step from destabilising training.

In [ ]:
nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)   # cap explosions
print("clipped to a safe norm")

## Visualize the data & results

_In a real deep sigmoid network on digits, how small is the weight update by the time it reaches the first layer?_

### Step 1 — Train one step of a deep sigmoid net on digits

We use scikit-learn's `MLPClassifier` with 8 hidden sigmoid (`logistic`) layers on the digit images, scaled to 0–1. `warm_start` plus `max_iter=1` lets us run exactly one optimisation step at a time so we can inspect what each step does to every layer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier

digits = load_digits()
X = digits.data / 16.0
y = digits.target

deep = MLPClassifier(hidden_layer_sizes=(32,) * 8, activation="logistic",
                     solver="sgd", max_iter=1, warm_start=True,
                     random_state=0, learning_rate_init=0.5)
deep.partial_fit(X, y, classes=np.unique(y))

### Step 2 — Measure how much each layer actually moved

We snapshot every layer's weight matrix, take one more training step, then measure the mean absolute change per layer. This is the real weight *update* SGD applied — and we expect the layers nearest the input to have barely moved, because their gradients vanished.

In [ ]:
before = [w.copy() for w in deep.coefs_]
deep.partial_fit(X, y)
updates = [np.abs(after - prev).mean() for after, prev in zip(deep.coefs_, before)]

### Step 3 — Plot the update size per layer

We order the layers output-first and plot the update size on a log scale. The bars fall off sharply toward the input layers (`L1 (in)`): the first layers receive an update orders of magnitude smaller than the last — the vanishing gradient, measured on a real network.

In [ ]:
labels = ["L9 (out)", "L8", "L7", "L6", "L5", "L4", "L3", "L2", "L1 (in)"]
values = updates[::-1]                    # output-first
colors = ["#7ee787", "#7ee787", "#4ea1ff", "#4ea1ff", "#ffb454",
          "#ffb454", "#ff7b72", "#ff7b72", "#ff7b72"]

plt.bar(labels, values, color=colors)
plt.title("Real mean weight-update per layer, 8-layer sigmoid digit net")
plt.yscale("log")
plt.show()